In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.GoldHelper

In [0]:
from pyspark.sql import functions as F 

In [0]:
target_table=f"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
drivers_silver_table=f"{catalog_name}.{silver_schema}.drivers"
nationality_table=f"{catalog_name}.{gold_schema}.nationality_continent"


In [0]:

df_drivers = (spark.table(drivers_silver_table)
              .filter(F.col("batch_id") == v_batch_id)
                )
df_nationality = spark.table(nationality_table)



In [0]:
df_drivers_dims=(
    df_drivers
    .join(df_nationality,
        df_drivers.nationality == df_nationality.Nationality,
        "left")
    .select(
        df_drivers.driver_id.alias('driver_id'),
        df_drivers.driver_name.alias('driver_name'),
        df_drivers.date_of_birth.alias('date_of_birth'),
        df_drivers.nationality.alias('nationality'),
        df_nationality.Continent.alias('Continent'),
        df_drivers.batch_id.alias('batch_id'),
    )
)



In [0]:
display(df_drivers_dims.select('nationality').distinct())

In [0]:
# df_drivers_dims.write.mode("overwrite").saveAsTable(target_table)

In [0]:
write_to_gold(
    input_df=df_drivers_dims,
    target_table=target_table,
    merge_condition="t.driver_id = s.driver_id",
    columns_to_update=[
        "driver_name",
        "date_of_birth",
        "nationality",
        "Continent",
        "batch_id"
    ]
)


In [0]:
display(spark.table(target_table))